# Walmart Hurricane Analytics

**DS2002 Midterm Project | Fall 2026 | University of Virginia**

**Team:** HEJ

**Members:** Evie McMahan, Julia McPhillips, Honor Dold

## 1. Data Cleaning

Cleaning functions, the SKU mapping table, and the decision log.

Graded on a clear pipeline: duplicates, missing values, SKU consolidation, type fixes.

Done when one function turns raw files into clean tables. The log lists every decision and rows touched.

**Data problems:**
- Duplicate records
- Missing values written several ways: empty, NULL, N/A, NaN
- Dates and times in different formats
- The same product under several SKUs and names
- Inconsistent capitalization and spacing
- Category names that don't match
- Numbers stored as text, prices with $ signs
- Negative quantities
- Store ids written different ways across files
- Weather data in a different timezone than sales

### Profile the data

In [127]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd, os

def find_data(filename, folders=('data', '../data', '/kaggle/input', '/content')):
    for folder in folders:
        path = os.path.join(folder, filename)
        if os.path.exists(path):
            return path
    return None

def profile(df, name='dataset'):
    print(f'=== {name} ===')
    print(f'{len(df):,} rows x {len(df.columns)} columns')
    print()
    summary = pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'nulls': df.isnull().sum(),
        'null_pct': (100 * df.isnull().mean()).round(1),
        'distinct': df.nunique(),
    })
    print(summary)
    print()
    print('exact duplicate rows:', df.duplicated().sum())
    return summary

After profiling the walmart_transactions data, I need to pull in other data files to compare these values to what the real counts are.

In [128]:
files = { # Group all the other files together to loop over
    'transactions': 'walmart_transactions.csv',
    'stores': 'store_locations.csv',
    'catalog': 'product_catalog.csv',
    'distribution_centers': 'distribution_centers.csv',
    
}

raw = {}
for name, filename in files.items(): # Loop through each file
    file_path = find_data(filename)
    if file_path: # Use the code that was given in lecture to read the file and profile it if found
        raw[name] = pd.read_csv(file_path) 
        profile(raw[name], name)
        print()
    else:
        print(f'Could not find {filename}. Upload the data/ folder in Kaggle or Colab, or run this inside the repo.')

tx_raw = raw['transactions']

=== transactions ===
25,562 rows x 8 columns

                dtype  nulls  null_pct  distinct
transaction_id    str      0       0.0     25532
store_id          str      0       0.0        60
timestamp         str    886       3.5     23952
sku               str      0       0.0        45
product_name      str      0       0.0        63
category          str      0       0.0        37
quantity        int64      0       0.0        11
unit_price        str      0       0.0      3668

exact duplicate rows: 30

=== stores ===
14 rows x 8 columns

              dtype  nulls  null_pct  distinct
store_id        str      0       0.0        14
store_name      str      0       0.0        12
city            str      0       0.0        12
state           str      0       0.0         3
zip_code      int64      0       0.0        12
latitude    float64      1       7.1        11
longitude   float64      1       7.1        11
region          str      0       0.0         7

exact duplicate rows: 0

=

In [129]:
def spelling_noise(df, cols): # Check for basic spelling and formatting noise
    for c in cols: # Loop through columns
        raw = df[c].dropna().astype(str)
        norm = raw.str.strip().str.lower().str.replace(r'[^a-z0-9]', '', regex=True) # Normalize values
        print(f'{c:15s} raw distinct: {raw.nunique():3d} | after normalizing: {norm.nunique():3d}') # Compare raw distinct values to normalized values

spelling_noise(raw['transactions'], ['store_id', 'product_name', 'category', 'sku'])
spelling_noise(raw['stores'], ['store_id', 'state', 'city'])
spelling_noise(raw['catalog'], ['product_name', 'category', 'subcategory'])

store_id        raw distinct:  60 | after normalizing:  12
product_name    raw distinct:  63 | after normalizing:  49
category        raw distinct:  37 | after normalizing:  18
sku             raw distinct:  45 | after normalizing:  45
store_id        raw distinct:  14 | after normalizing:  12
state           raw distinct:   3 | after normalizing:   3
city            raw distinct:  12 | after normalizing:  12
product_name    raw distinct:  38 | after normalizing:  34
category        raw distinct:  27 | after normalizing:  15
subcategory     raw distinct:  19 | after normalizing:  19


In [130]:
# Look closer at whichever columns worried you most
tx = tx_raw.copy()

if 'transactions' in raw:
    print('sample of sku values:')
    print(tx['sku'].value_counts().head(10))
    print()
    print('sample of raw quantity values:')
    print(tx['quantity'].astype(str).value_counts().head(10))

sample of sku values:
sku
TARP-HD         766
LED-FLASH       732
FLASH-01        728
HDTARP-01       701
CHRG-PORT       684
CANDLE-EMRG     664
CNDL-EM         652
BEER-24         643
FIRSTAID-KIT    637
PORTCHARGER     632
Name: count, dtype: int64

sample of raw quantity values:
quantity
1     10268
2      9290
3      3745
4      1343
5       528
6       258
-1       54
-2       53
-3       15
-5        4
Name: count, dtype: int64


The store profile shows that there are 14 rows in the store file, but there are 60 distinct `store_id`s in the transactions profile. This means there is spelling and formatting noise, since there should only be 12 distinct `store_id`s (the store file has 14 rows, but 2 are duplicates).

The catalog profile shows that there are 45 distinct SKUs, which matches the 45 distinct `sku`s in transactions. Catalog also shows that there are 38 distinct product names, but there are 63 distinct `product_name` values in transactions. Finally, there are 27 distinct categories in catalog, but there are 37 in the transactions data.

So far, within the transactions dataset I need to remove the noise in `store_id`, `product_name`, and `category`. After cleaning, `store_id` should have 12 distinct values (the store file has 14 rows, but 2 are duplicates). The `sku` column is clean (45 valid codes), but the catalog has only 38 product names for 45 SKUs, so several SKUs are the same product.

We will also have to do formatting normalizing for some of the rows based on the basic normalizing check.

On data types, we will have to address `unit_price`, which is currently a string, along with some others. `quantity` also has negative values. We also have to address the null values that are shown in the profile. 

### Decision Log

In [131]:
decision_log = [] # Create decision log

def log(step, column, decision, rows_before=None, rows_after=None):
    decision_log.append({
        'step': step,
        'column': column,
        'decision': decision,
        'rows_before': rows_before,
        'rows_after': rows_after,
    })

# After each cleaning step, call log(...). 
# View the whole log with: pd.DataFrame(decision_log)

### #1: Trim inconsistent spacing

In [132]:
def trim_text(df, name):
    """Trim and collapse spaces in every string columns; blank strings become missing."""
    out = df.copy()
    text_cols = [c for c in out.columns if not pd.api.types.is_numeric_dtype(out[c])]
    changed = {}
    for c in text_cols: # Go through the cleaning process
        s = out[c]
        cleaned = s.str.strip().str.replace(r'\s+', ' ', regex=True).replace('', np.nan)
        changed[c] = int((s.fillna('') != cleaned.fillna('')).sum())
        out[c] = cleaned
    fixed = {k: v for k, v in changed.items() if v}
    log(f'{name}: trim spaces', ', '.join(text_cols), # Log the decision
        f'trimmed/collapsed spaces, blanks set to missing; values changed: {fixed}',
        len(df), len(out))
    return out

trimmed = {}
for name, df in raw.items(): # Trim each string column in each dataframe dataframe
    trimmed[name] = trim_text(df, name)

pd.DataFrame(decision_log)

,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, product_name, category, unit_price","trimmed/collapsed spaces, blanks set to missing; values changed: {'store_id': 738, 'product_name': 341, 'unit_price': 1303}",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missing; values changed: {'region': 1}",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supplier, package_size","trimmed/collapsed spaces, blanks set to missing; values changed: {}",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missing; values changed: {}",5,5


Trimming the duplicate spacing in every row in a string column impacted only `store_id`, `product_name`, `unit_price`, and `region`. All rows were preserved.

### #2: Deal with missing values: empty, NULL, N/A, NaN

In [133]:
missing_types = ['', 'NULL', 'N/A', 'NaN', 'nan'] # Different types of missing values

def check_missing(path): 
    text = pd.read_csv(path, dtype=str, keep_default_na=False)   # Read as plain text
    for col in text.columns: # Loop through each column
        values = text[col].str.strip() # Remove spaces
        missing = values[values.isin(missing_types)] # Find missing values
        if len(missing) > 0: # If there are missing values
            print(f'  {col}: {len(missing)} missing, written as:',
                  missing.replace('', '(blank)').value_counts().to_dict())

for name, filename in files.items():
    print(name)
    check_missing(find_data(filename))

transactions
  timestamp: 886 missing, written as: {'NULL': 188, 'N/A': 181, '(blank)': 176, 'NaN': 173, 'nan': 168}
stores
  latitude: 1 missing, written as: {'(blank)': 1}
  longitude: 1 missing, written as: {'(blank)': 1}
catalog
  subcategory: 13 missing, written as: {'(blank)': 13}
  unit_cost: 5 missing, written as: {'(blank)': 5}
distribution_centers
  cost_per_shipment: 1 missing, written as: {'(blank)': 1}


DECIDE WHAT TO DO WITH MISSING VALUES!

### #3: DataType Formatting

Based on my profiling above, I saw that I need to clean the datatypes for `unit_price` and `timestamps` in transactions and `surge_capacity` in distribution centers.

In [ ]:
def to_price(s):
    """'$3.38' -> 3.38"""
    out = pd.to_numeric(s.str.replace(r'[$,]', '', regex=True), errors='coerce')
    assert out.notna().all(), 'some prices failed to convert'
    return out

def to_datetime(s):
    """Parse all formats; values that can't be parsed become missing."""
    return pd.to_datetime(s, errors='coerce', format='mixed')

def to_capacity(s):
    """'~750' -> 750"""
    return pd.to_numeric(s.astype(str).str.replace(r'[^0-9.]', '', regex=True))

def fix_types(tables): # Fix all the types
    out = dict(tables)

    t = out['transactions'].copy()
    t['unit_price'] = to_price(t['unit_price']) # Fix unit price in transactions
    t['timestamp'] = to_datetime(t['timestamp']) # Fix timestamp in transactions
    t['ts_missing'] = t['timestamp'].isna()
    log('transactions: types', 'unit_price, timestamp',
        'unit_price to float (removed $); timestamp parsed to datetime; missing timestamps flagged in ts_missing',
        len(t), len(t))
    out['transactions'] = t

    d = out['distribution_centers'].copy()
    d['surge_capacity_approx'] = d['surge_capacity'].astype(str).str.contains('~')
    d['surge_capacity'] = to_capacity(d['surge_capacity']) # Fix surge capacity in distribution centers
    log('distribution centers: types', 'surge_capacity', 'removed ~, converted to number; approximate value flagged', len(d), len(d))
    out['distribution_centers'] = d
    return out

cleaned_type = fix_types(trimmed) # Clean types on trimmed dataframes
cleaned_type['transactions'].dtypes
pd.DataFrame(decision_log)


,step,column,decision,rows_before,rows_after
0,transactions: trim spaces,"transaction_id, store_id, timestamp, sku, product_name, category, unit_price","trimmed/collapsed spaces, blanks set to missing; values changed: {'store_id': 738, 'product_name': 341, 'unit_price': 1303}",25562,25562
1,stores: trim spaces,"store_id, store_name, city, state, region","trimmed/collapsed spaces, blanks set to missing; values changed: {'region': 1}",14,14
2,catalog: trim spaces,"sku, product_name, category, subcategory, supplier, package_size","trimmed/collapsed spaces, blanks set to missing; values changed: {}",45,45
3,distribution_centers: trim spaces,"dc_id, dc_name, city, state, surge_capacity","trimmed/collapsed spaces, blanks set to missing; values changed: {}",5,5
4,transactions: types,"unit_price, timestamp",unit_price to float (removed $); timestamp parsed to datetime; missing timestamps flagged in ts_missing,25562,25562
5,distribution centers: types,surge_capacity,"removed ~, converted to number; approximate value flagged",5,5


## 2. Weather API

API call with retries, parsing, timezone alignment, and a checked join to sales.

Graded on a working call, timezones handled, and a join you checked.

Done when one function returns daily weather. The join passes a row-count check. Base URL cited.

## 3. The Five Questions

The five answers: code, a number, and a written finding each. Q5 uses SQLite.

Graded on a code, a visualization, and a written explanation for each one.

Done when every question has code, a number, and a paragraph.

## 4. Visualizations

At least six plots with message titles, the six-minute talk, and the README.

Graded on at least six plots with clear titles, labels, and legends.

Done when six labeled plots. The talk has been rehearsed once.

## 5. Reflection

Graded on five questions, one solid paragraph each, every member contributes.